In [ ]:
%pip install -q transformers==4.46.3 huggingface-hub==0.36.0

In [ ]:
"""
RobustLungAI — Respiratory Sound Classification & Interpretability

Modified for Google Colab:
- Automatically downloads the fine-tuned checkpoint from Hugging Face.
- Uses the public repository: praveshsubba/robustlungai-ast
- Keeps strict state-dictionary loading.
- Supports four classes: normal, crackle, wheeze, both.
- Includes attention rollout, spectrogram overlay, and window timeline.
- Provides a Gradio interface with an optional public share URL.

Research prototype only — not a medical device.
"""


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import sys
import html
import threading
import traceback
from pathlib import Path

import numpy as np
import gradio as gr
import librosa
import torch

from huggingface_hub import hf_hub_download
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from scipy.signal import butter, sosfiltfilt
from transformers import ASTFeatureExtractor, ASTForAudioClassification


# ============================================================
# 2. CONFIGURATION
# ============================================================

# Hugging Face repository containing your fine-tuned model.
HF_REPO_ID = "praveshsubba/robustlungai-ast"
HF_CHECKPOINT_FILENAME = "ast_lung_fp16.pt"

# Original AST pretrained checkpoint.
AST_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"

# Colab-friendly application directory.
APP_DIR = Path("/content")
REPO_ROOT = APP_DIR

# Optional override:
# If ROBUSTLUNG_CHECKPOINT points to a local checkpoint, use it.
# Otherwise, download the checkpoint from Hugging Face.
LOCAL_CHECKPOINT_OVERRIDE = os.environ.get(
    "ROBUSTLUNG_CHECKPOINT", ""
).strip()

if LOCAL_CHECKPOINT_OVERRIDE:
    CHECKPOINT_PATH = Path(
        LOCAL_CHECKPOINT_OVERRIDE
    ).expanduser()
    print("Using explicitly configured local checkpoint.")
else:
    print("Downloading RobustLungAI checkpoint from Hugging Face...")

    try:
        CHECKPOINT_PATH = Path(
            hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_CHECKPOINT_FILENAME,
                repo_type="model",
            )
        )

        print("Checkpoint downloaded or found in the local cache.")
        print("Checkpoint path:", CHECKPOINT_PATH)
        print(
            f"Checkpoint size: "
            f"{CHECKPOINT_PATH.stat().st_size / (1024 ** 2):.2f} MiB"
        )

    except Exception as exc:
        raise RuntimeError(
            "Could not obtain the fine-tuned RobustLungAI checkpoint.\n"
            f"Repository: {HF_REPO_ID}\n"
            f"Filename: {HF_CHECKPOINT_FILENAME}\n"
            "Check that the Hugging Face repository is public and that "
            "the checkpoint filename matches exactly.\n"
            f"Original error: {exc}"
        ) from exc


# Audio configuration.
TARGET_SR = 16000
LOWCUT = 50
HIGHCUT = 8000

# Sliding-window configuration.
WINDOW_S = 8.0
STRIDE_S = 4.0
MAX_AUDIO_S = 120.0
MIN_AUDIO_S = 0.5
BATCH_SIZE = 4

# AST spectrogram and patch configuration.
N_MELS = 128
N_FRAMES_PAD = 1024
PATCH = 16
PATCH_STRIDE = 10

F_PATCHES = (
    (N_MELS - PATCH) // PATCH_STRIDE + 1
)
T_PATCHES = (
    (N_FRAMES_PAD - PATCH) // PATCH_STRIDE + 1
)

N_SPECIAL_TOKENS = 2
FRAME_SHIFT_S = 0.010

# Four target classes.
LABELS = [
    "normal",
    "crackle",
    "wheeze",
    "both",
]

LABEL_INFO = {
    "normal": (
        "No adventitious sounds",
        "#10b981",
    ),
    "crackle": (
        "Discontinuous, popping sounds",
        "#f59e0b",
    ),
    "wheeze": (
        "Continuous, high-pitched sounds",
        "#3b82f6",
    ),
    "both": (
        "Crackles and wheezes together",
        "#ef4444",
    ),
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IS_GRADIO_6 = (
    int(gr.__version__.split(".")[0]) >= 6
)

print("=" * 65)
print("RobustLungAI")
print("=" * 65)
print("Gradio:", gr.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)


# ============================================================
# 3. CHECKPOINT DISCOVERY
# ============================================================

def find_checkpoint():
    """Return the configured, downloaded fine-tuned checkpoint."""

    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(
            "Fine-tuned checkpoint not found.\n"
            f"Expected checkpoint: {CHECKPOINT_PATH}\n"
            "Check the Hugging Face repository and filename."
        )

    return CHECKPOINT_PATH


# ============================================================
# 4. AUDIO PREPROCESSING
# ============================================================

def bandpass_filter(signal, sr, order=5):
    """
    Apply a Butterworth band-pass filter.

    Low cutoff: 50 Hz.
    High cutoff: up to 8000 Hz, limited by Nyquist.
    """

    highcut = min(
        HIGHCUT,
        (sr / 2.0) * 0.99,
    )

    if LOWCUT >= highcut:
        raise ValueError(
            f"Sample rate {sr} Hz is too low "
            "for the band-pass filter."
        )

    sos = butter(
        order,
        [LOWCUT, highcut],
        btype="band",
        fs=sr,
        output="sos",
    )

    return sosfiltfilt(
        sos,
        signal,
    ).astype(np.float32)


def pad_or_tile(signal, target_length):
    """Fit a waveform to the target length."""

    if len(signal) >= target_length:
        return signal[:target_length].astype(np.float32)

    if len(signal) == 0:
        raise ValueError("The audio signal is empty.")

    repeats = int(
        np.ceil(target_length / len(signal))
    )

    return np.tile(
        signal,
        repeats,
    )[:target_length].astype(np.float32)


def n_real_frames(n_samples, sr=TARGET_SR):
    """Calculate the number of real spectrogram frames."""

    frame_length = int(0.025 * sr)
    frame_shift = int(0.010 * sr)

    frames = (
        1
        + (n_samples - frame_length) // frame_shift
    )

    return int(
        min(
            max(frames, 1),
            N_FRAMES_PAD,
        )
    )


def make_windows(audio, sr=TARGET_SR):
    """
    Split audio into overlapping 8-second windows
    with a 4-second stride.
    """

    win = int(WINDOW_S * sr)
    stride = int(STRIDE_S * sr)

    if len(audio) <= win:
        return [
            {
                "start_s": 0.0,
                "end_s": len(audio) / sr,
                "n_real": len(audio),
                "waveform": pad_or_tile(
                    audio,
                    win,
                ),
            }
        ]

    starts = list(
        range(
            0,
            len(audio) - win + 1,
            stride,
        )
    )

    if starts[-1] + win < len(audio):
        starts.append(len(audio) - win)

    windows = []

    for start in starts:
        windows.append(
            {
                "start_s": start / sr,
                "end_s": start / sr + WINDOW_S,
                "n_real": win,
                "waveform": audio[
                    start:start + win
                ].astype(np.float32),
            }
        )

    return windows


# ============================================================
# 5. ATTENTION ROLLOUT
# ============================================================

def attention_rollout(attention_matrices):
    """
    Combine attention matrices from transformer layers.

    Residual connections are approximated by adding the
    identity matrix before normalization.
    """

    if not attention_matrices:
        raise ValueError(
            "No attention matrices were returned."
        )

    n = attention_matrices[0].shape[0]

    identity = np.eye(
        n,
        dtype=np.float64,
    )

    rollout = identity.copy()

    for attention in attention_matrices:
        aug = (
            0.5 * attention.astype(np.float64)
            + 0.5 * identity
        )

        aug /= (
            aug.sum(
                axis=-1,
                keepdims=True,
            ) + 1e-12
        )

        rollout = aug @ rollout

    return rollout


def rollout_to_patch_grid(rollout):
    """
    Extract the patch-token attention scores and reshape
    them into a two-dimensional frequency-time grid.
    """

    scores = 0.5 * (
        rollout[0, N_SPECIAL_TOKENS:]
        + rollout[1, N_SPECIAL_TOKENS:]
    )

    expected = F_PATCHES * T_PATCHES

    if scores.size != expected:
        raise ValueError(
            f"Expected {expected} patch scores, "
            f"received {scores.size}. "
            "Check the AST architecture and token layout."
        )

    return scores.reshape(
        F_PATCHES,
        T_PATCHES,
    )


def patch_grid_to_spectrogram(grid):
    """Map patch attention scores onto spectrogram bins."""

    canvas = np.zeros(
        (N_MELS, N_FRAMES_PAD)
    )

    counts = np.zeros_like(canvas)

    for f in range(F_PATCHES):
        for t in range(T_PATCHES):
            fs = f * PATCH_STRIDE
            ts = t * PATCH_STRIDE

            canvas[
                fs:fs + PATCH,
                ts:ts + PATCH,
            ] += grid[f, t]

            counts[
                fs:fs + PATCH,
                ts:ts + PATCH,
            ] += 1

    out = np.divide(
        canvas,
        counts,
        out=np.zeros_like(canvas),
        where=counts > 0,
    )

    rows = np.where(
        counts.max(axis=1) > 0
    )[0]

    cols = np.where(
        counts.max(axis=0) > 0
    )[0]

    if len(rows):
        out[rows[-1] + 1:, :] = out[rows[-1], :]

    if len(cols):
        out[:, cols[-1] + 1:] = (
            out[:, cols[-1]][:, None]
        )

    return out


def normalize01(array):
    """Normalize an array to the interval [0, 1]."""

    array = np.asarray(
        array,
        dtype=np.float32,
    )

    return (
        (array - array.min())
        / (array.max() - array.min() + 1e-12)
    )


# ============================================================
# 6. MODEL LOADING
# ============================================================

_model = None
_feature_extractor = None
_model_lock = threading.Lock()


def read_state_dict(path):
    """
    Load the PyTorch checkpoint.

    Supports a direct state dictionary or common wrappers
    such as state_dict and model_state_dict.
    """

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )

    except TypeError:
        # Compatibility with older PyTorch versions.
        checkpoint = torch.load(
            path,
            map_location="cpu",
        )

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Checkpoint must contain a PyTorch state dictionary."
        )

    for key in (
        "state_dict",
        "model_state_dict",
        "model",
    ):
        if (
            key in checkpoint
            and isinstance(checkpoint[key], dict)
        ):
            checkpoint = checkpoint[key]
            break

    state = {}

    for key, value in checkpoint.items():
        if key.startswith("module."):
            key = key[len("module."):]

        # Convert floating weights to float32 before loading.
        # The model can subsequently be placed on the chosen device.
        if (
            torch.is_tensor(value)
            and value.is_floating_point()
        ):
            value = value.float()

        state[key] = value

    return state


def get_model():
    """Load the AST model once, using a thread-safe lock."""

    global _model, _feature_extractor

    with _model_lock:

        if _model is not None:
            return _model, _feature_extractor

        checkpoint_path = find_checkpoint()

        print("Loading AST feature extractor...")

        feature_extractor = (
            ASTFeatureExtractor.from_pretrained(
                AST_CHECKPOINT
            )
        )

        id2label = dict(
            enumerate(LABELS)
        )

        model_args = {
            "num_labels": len(LABELS),
            "id2label": id2label,
            "label2id": {
                label: index
                for index, label in id2label.items()
            },
            "ignore_mismatched_sizes": True,
        }

        try:
            model = ASTForAudioClassification.from_pretrained(
                AST_CHECKPOINT,
                attn_implementation="eager",
                **model_args,
            )

        except TypeError:
            # Compatibility fallback for older Transformers.
            model = ASTForAudioClassification.from_pretrained(
                AST_CHECKPOINT,
                **model_args,
            )

        print(
            "Loading fine-tuned weights:",
            checkpoint_path,
        )

        state = read_state_dict(
            checkpoint_path
        )

        load_result = model.load_state_dict(
            state,
            strict=True,
        )

        if (
            load_result.missing_keys
            or load_result.unexpected_keys
        ):
            raise RuntimeError(
                "Checkpoint mismatch: "
                f"missing={load_result.missing_keys}, "
                f"unexpected={load_result.unexpected_keys}"
            )

        print(
            f"Loaded all {len(state)} checkpoint entries strictly."
        )

        _model = model.to(DEVICE).eval()
        _feature_extractor = feature_extractor

        print("Model ready.")

        return _model, _feature_extractor


# ============================================================
# 7. INFERENCE
# ============================================================

@torch.no_grad()
def predict_probabilities(
    model,
    feature_extractor,
    waveforms,
):
    """
    Predict class probabilities in batches.

    Attention outputs are not requested here, which helps
    reduce memory consumption during normal inference.
    """

    outputs = []

    for i in range(
        0,
        len(waveforms),
        BATCH_SIZE,
    ):
        batch = waveforms[
            i:i + BATCH_SIZE
        ]

        inputs = feature_extractor(
            batch,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
        )["input_values"].to(DEVICE)

        logits = model(
            input_values=inputs
        ).logits

        probabilities = (
            logits.float()
            .softmax(dim=-1)
            .cpu()
            .numpy()
        )

        outputs.append(probabilities)

    return np.concatenate(
        outputs,
        axis=0,
    )


@torch.no_grad()
def explain_window(
    model,
    feature_extractor,
    waveform,
):
    """Generate attention rollout for one selected window."""

    inputs = feature_extractor(
        waveform,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )["input_values"].to(DEVICE)

    outputs = model(
        input_values=inputs,
        output_attentions=True,
    )

    if outputs.attentions is None:
        raise RuntimeError(
            "The model returned no attentions. "
            "Load it with attn_implementation='eager'."
        )

    attention_matrices = [
        attention[0]
        .float()
        .mean(dim=0)
        .cpu()
        .numpy()
        for attention in outputs.attentions
    ]

    rollout = attention_rollout(
        attention_matrices
    )

    patch_grid = rollout_to_patch_grid(
        rollout
    )

    heatmap = patch_grid_to_spectrogram(
        patch_grid
    )

    # ASTFeatureExtractor returns log-mel input_values:
    # frames x mel bins for the selected waveform.
    spectrogram = (
        inputs[0]
        .float()
        .cpu()
        .numpy()
    )

    return heatmap, spectrogram


# ============================================================
# 8. FIGURE HELPERS
# ============================================================

MUTED = "#64748b"


def _style_axis(ax):
    """Style axes for light and dark themes."""

    ax.set_facecolor("none")

    for spine in ax.spines.values():
        spine.set_color(MUTED)
        spine.set_linewidth(0.6)

    ax.tick_params(
        colors=MUTED,
        labelsize=9,
    )

    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    ax.title.set_color(MUTED)


def _to_array(fig):
    """Convert a Matplotlib figure into a NumPy image."""

    canvas = FigureCanvasAgg(fig)
    canvas.draw()

    image = np.asarray(
        canvas.buffer_rgba()
    ).copy()

    return image


def render_interpretability(
    spec,
    heatmap,
    n_frames,
    start_s,
    end_s,
):
    """Render the spectrogram and attention overlay."""

    fig = Figure(
        figsize=(12, 4.2),
        dpi=140,
        facecolor=(0, 0, 0, 0),
    )

    ax1, ax2 = fig.subplots(
        1,
        2,
    )

    extent = [
        0,
        n_frames * FRAME_SHIFT_S,
        0,
        N_MELS,
    ]

    view = spec[:n_frames].T

    ax1.imshow(
        view,
        origin="lower",
        aspect="auto",
        cmap="magma",
        extent=extent,
    )

    ax1.set_title(
        f"Log-mel spectrogram · "
        f"{start_s:.1f}–{end_s:.1f} s",
        fontsize=11,
    )

    ax2.imshow(
        view,
        origin="lower",
        aspect="auto",
        cmap="gray",
        extent=extent,
    )

    image = ax2.imshow(
        normalize01(
            heatmap[:, :n_frames]
        ),
        origin="lower",
        aspect="auto",
        cmap="jet",
        alpha=0.5,
        extent=extent,
    )

    ax2.set_title(
        "Attention rollout overlay",
        fontsize=11,
    )

    for ax in (ax1, ax2):
        _style_axis(ax)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Mel bin")

    colorbar = fig.colorbar(
        image,
        ax=ax2,
        fraction=0.046,
        pad=0.02,
    )

    colorbar.set_label(
        "Relative attention",
        color=MUTED,
        fontsize=9,
    )

    colorbar.ax.tick_params(
        colors=MUTED,
        labelsize=8,
    )

    colorbar.outline.set_edgecolor(
        MUTED
    )

    fig.tight_layout()

    return _to_array(fig)


def render_timeline(
    windows,
    probs,
    best_index,
):
    """Render the predicted class probabilities over time."""

    fig = Figure(
        figsize=(12, 3.6),
        dpi=140,
        facecolor=(0, 0, 0, 0),
    )

    ax = fig.subplots()

    centers = np.array([
        (
            window["start_s"]
            + window["end_s"]
        ) / 2
        for window in windows
    ])

    for i, label in enumerate(LABELS):
        ax.plot(
            centers,
            probs[:, i],
            marker="o",
            markersize=5,
            linewidth=2,
            color=LABEL_INFO[label][1],
            label=label.capitalize(),
        )

    best_window = windows[best_index]

    ax.axvspan(
        best_window["start_s"],
        best_window["end_s"],
        color="#14b8a6",
        alpha=0.12,
        label="Selected window",
    )

    if len(windows) == 1:
        ax.set_xlim(
            best_window["start_s"] - 1,
            best_window["end_s"] + 1,
        )

    ax.set_ylim(
        -0.02,
        1.02,
    )

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Probability")

    ax.set_title(
        "Window-level class probabilities",
        fontsize=11,
    )

    ax.grid(
        alpha=0.2,
        color=MUTED,
    )

    _style_axis(ax)

    legend = ax.legend(
        ncol=5,
        fontsize=9,
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
    )

    for text_item in legend.get_texts():
        text_item.set_color(MUTED)

    fig.tight_layout()

    return _to_array(fig)


# ============================================================
# 9. RESULT AND STATUS HTML
# ============================================================

EMPTY_HTML = """
<div class="result-card empty">
  <div class="empty-icon">🫁</div>
  <div class="empty-title">No analysis yet</div>
  <div class="empty-sub">
    Upload a lung-sound recording and press
    <b>Analyze recording</b>.
  </div>
</div>
"""


def error_html(message):
    """Create an HTML error card."""

    return f"""
    <div class="result-card error">
      <div class="result-kicker">Analysis failed</div>
      <div class="error-msg">{html.escape(message)}</div>
      <div class="result-sub">
        See the notebook output for the full traceback.
      </div>
    </div>
    """


def result_html(
    predicted,
    probs,
    n_windows,
    duration_s,
    best_window,
    note,
):
    """Create the recording-level prediction card."""

    description, color = LABEL_INFO[predicted]

    confidence = (
        float(probs[LABELS.index(predicted)])
        * 100
    )

    note_html = (
        f'<div class="result-note">'
        f'{html.escape(note)}</div>'
        if note
        else ""
    )

    return f"""
    <div class="result-card">
      <div class="result-kicker">Predicted class</div>

      <div class="result-main">
        <span class="badge"
          style="background:{color}22;color:{color};
          border-color:{color}66">
          {html.escape(predicted.capitalize())}
        </span>

        <span class="result-conf">
          {confidence:.1f}% mean probability
        </span>
      </div>

      <div class="result-sub">
        {html.escape(description)}
      </div>

      <div class="stat-grid">
        <div class="stat">
          <div class="stat-v">{n_windows}</div>
          <div class="stat-l">Windows</div>
        </div>

        <div class="stat">
          <div class="stat-v">{duration_s:.1f}s</div>
          <div class="stat-l">Duration analyzed</div>
        </div>

        <div class="stat">
          <div class="stat-v">
            {best_window['start_s']:.1f}–{best_window['end_s']:.1f}s
          </div>
          <div class="stat-l">Explained window</div>
        </div>
      </div>

      {note_html}

      <div class="result-disclaimer">
        <b>Research prototype — not a medical device.</b>
        Probabilities are averaged over overlapping windows.
        The model was evaluated on annotated breath cycles;
        whole-recording aggregation is a demo convenience,
        not a validated diagnostic system.
      </div>
    </div>
    """


# ============================================================
# 10. RECORDING-LEVEL PIPELINE
# ============================================================

def run(
    audio_path,
    progress=gr.Progress(),
):
    """Complete audio preprocessing, inference and explanation."""

    if audio_path is None:
        return (
            None,
            error_html(
                "Please upload a recording first."
            ),
            None,
            None,
        )

    try:
        progress(
            0.05,
            desc="Loading audio",
        )

        raw, sr = librosa.load(
            audio_path,
            sr=None,
            mono=True,
        )

        raw = np.asarray(
            raw,
            dtype=np.float32,
        )

        if len(raw) < int(MIN_AUDIO_S * sr):
            raise ValueError(
                f"Recording is too short "
                f"(minimum {MIN_AUDIO_S:.1f} s)."
            )

        progress(
            0.15,
            desc="Filtering and resampling",
        )

        audio = bandpass_filter(
            raw,
            sr,
        )

        if sr != TARGET_SR:
            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=TARGET_SR,
            )

        audio = np.asarray(
            audio,
            dtype=np.float32,
        )

        note = ""

        max_len = int(
            MAX_AUDIO_S * TARGET_SR
        )

        if len(audio) > max_len:
            audio = audio[:max_len]

            note = (
                "Recording truncated to the first "
                f"{MAX_AUDIO_S:.0f} seconds for this demo."
            )

        windows = make_windows(
            audio
        )

        progress(
            0.30,
            desc="Loading model",
        )

        model, feature_extractor = get_model()

        progress(
            0.45,
            desc=f"Classifying {len(windows)} window(s)",
        )

        probs = predict_probabilities(
            model,
            feature_extractor,
            [
                window["waveform"]
                for window in windows
            ],
        )

        # Mean window-level probabilities.
        mean_probs = probs.mean(
            axis=0
        )

        pred_idx = int(
            mean_probs.argmax()
        )

        predicted = LABELS[pred_idx]

        # Explain the window most supportive of the
        # recording-level predicted class.
        best_index = int(
            probs[:, pred_idx].argmax()
        )

        best = windows[best_index]

        progress(
            0.75,
            desc="Computing attention rollout",
        )

        heatmap, spec = explain_window(
            model,
            feature_extractor,
            best["waveform"],
        )

        n_frames = n_real_frames(
            best["n_real"]
        )

        progress(
            0.92,
            desc="Rendering figures",
        )

        spec_image = render_interpretability(
            spec,
            heatmap,
            n_frames,
            best["start_s"],
            best["end_s"],
        )

        timeline_image = render_timeline(
            windows,
            probs,
            best_index,
        )

        label_output = {
            LABELS[i]: float(mean_probs[i])
            for i in range(len(LABELS))
        }

        card = result_html(
            predicted,
            mean_probs,
            len(windows),
            len(audio) / TARGET_SR,
            best,
            note,
        )

        return (
            label_output,
            card,
            spec_image,
            timeline_image,
        )

    except Exception as exc:
        traceback.print_exc()

        return (
            None,
            error_html(
                f"{type(exc).__name__}: {exc}"
            ),
            None,
            None,
        )


def clear_all():
    """Reset all interface outputs."""

    return (
        None,
        None,
        EMPTY_HTML,
        None,
        None,
    )


# ============================================================
# 11. THEME AND CSS
# ============================================================

APP_CSS = """
.gradio-container {
    max-width: 1200px !important;
    margin: 0 auto !important;
}

footer {
    display: none !important;
}

/* Header */
#app-header {
    background: linear-gradient(
        125deg,
        #0f2238 0%,
        #14405a 55%,
        #0f766e 100%
    );
    border-radius: 20px;
    padding: 34px 38px;
    margin-bottom: 16px;
    box-shadow: 0 10px 30px rgba(15, 34, 56, 0.25);
}

#app-header .hero-tag {
    display: inline-block;
    color: #a7f3d0;
    background: rgba(16, 185, 129, 0.15);
    border: 1px solid rgba(167, 243, 208, 0.35);
    border-radius: 999px;
    padding: 4px 12px;
    font-size: 12px;
    font-weight: 600;
    letter-spacing: 0.06em;
}

#app-header h1 {
    color: #ffffff !important;
    font-size: clamp(28px, 4.2vw, 42px);
    font-weight: 750;
    letter-spacing: -0.03em;
    margin: 14px 0 4px 0;
    line-height: 1.1;
}

#app-header h2 {
    color: #bfe3ea !important;
    font-size: clamp(15px, 2vw, 19px);
    font-weight: 500;
    margin: 0 0 14px 0;
}

#app-header p {
    color: #d8e8ef !important;
    line-height: 1.7;
    max-width: 760px;
    margin: 0;
}

#app-header .chips {
    margin-top: 18px;
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
}

#app-header .chip {
    color: #e6f4f8;
    background: rgba(255, 255, 255, 0.09);
    border: 1px solid rgba(255, 255, 255, 0.18);
    border-radius: 8px;
    padding: 4px 10px;
    font-size: 12px;
}

/* Research notice */
#research-note {
    border: 1px solid rgba(217, 119, 6, 0.55);
    background: rgba(245, 158, 11, 0.09);
    border-radius: 12px;
    padding: 12px 16px;
    margin-bottom: 16px;
    font-size: 13px;
    line-height: 1.65;
}

/* Cards */
.card {
    border: 1px solid var(--border-color-primary);
    border-radius: 16px;
    padding: 18px !important;
    background: var(--background-fill-primary);
}

.step-title {
    font-size: 13px;
    font-weight: 650;
    letter-spacing: 0.05em;
    text-transform: uppercase;
    color: var(--body-text-color-subdued);
    margin-bottom: 6px;
}

.helper-text {
    color: var(--body-text-color-subdued);
    font-size: 13px;
    line-height: 1.7;
}

.helper-grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 8px;
    margin: 4px 0;
}

.helper-grid div {
    background: var(--background-fill-secondary);
    border: 1px solid var(--border-color-primary);
    border-radius: 10px;
    padding: 8px 10px;
    font-size: 12px;
    color: var(--body-text-color-subdued);
}

.helper-grid b {
    display: block;
    color: var(--body-text-color);
    font-size: 13px;
}

#analyze-btn {
    min-height: 48px;
    border-radius: 11px;
    font-weight: 700;
    font-size: 15px;
}

#clear-btn {
    min-height: 48px;
    border-radius: 11px;
}

/* Results */
.result-card {
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 18px 20px;
    background: var(--background-fill-secondary);
    color: var(--body-text-color);
}

.result-kicker {
    font-size: 12px;
    font-weight: 650;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: var(--body-text-color-subdued);
}

.result-main {
    display: flex;
    align-items: center;
    gap: 12px;
    margin: 8px 0 4px;
    flex-wrap: wrap;
}

.badge {
    font-size: 24px;
    font-weight: 750;
    padding: 4px 16px;
    border-radius: 999px;
    border: 1px solid;
    letter-spacing: -0.01em;
}

.result-conf {
    font-size: 14px;
    color: var(--body-text-color-subdued);
}

.result-sub {
    font-size: 14px;
    color: var(--body-text-color-subdued);
    margin-bottom: 12px;
}

.stat-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 10px;
    margin: 10px 0 12px;
}

.stat {
    background: var(--background-fill-primary);
    border: 1px solid var(--border-color-primary);
    border-radius: 10px;
    padding: 10px 12px;
}

.stat-v {
    font-size: 17px;
    font-weight: 700;
}

.stat-l {
    font-size: 11px;
    color: var(--body-text-color-subdued);
    margin-top: 2px;
}

.result-note {
    font-size: 13px;
    margin-bottom: 10px;
    color: #b45309;
}

.result-disclaimer {
    font-size: 12px;
    line-height: 1.6;
    color: var(--body-text-color-subdued);
    border-top: 1px solid var(--border-color-primary);
    padding-top: 10px;
}

.result-card.empty {
    text-align: center;
    padding: 34px 20px;
}

.empty-icon {
    font-size: 34px;
    margin-bottom: 6px;
}

.empty-title {
    font-weight: 650;
    font-size: 16px;
}

.empty-sub {
    color: var(--body-text-color-subdued);
    font-size: 13px;
    margin-top: 4px;
}

.result-card.error {
    border-color: rgba(220, 38, 38, 0.5);
    background: rgba(220, 38, 38, 0.07);
}

.error-msg {
    font-family: ui-monospace, monospace;
    font-size: 13px;
    margin: 8px 0;
    word-break: break-word;
}

@media (max-width: 700px) {
    #app-header {
        padding: 22px 18px;
    }

    .card {
        padding: 12px !important;
    }

    .stat-grid {
        grid-template-columns: 1fr;
    }
}
"""


APP_THEME = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="blue",
    neutral_hue="slate",
    radius_size="lg",
    spacing_size="md",
    font=[
        gr.themes.GoogleFont("Inter"),
        "ui-sans-serif",
        "system-ui",
        "sans-serif",
    ],
)


HEADER_HTML = """
<div id="app-header">
  <span class="hero-tag">
    RESEARCH PROTOTYPE · AUDIO AI
  </span>

  <h1>RobustLungAI</h1>

  <h2>
    Respiratory sound classification &amp; interpretability
  </h2>

  <p>
    Analyze lung-sound recordings with a fine-tuned
    Audio Spectrogram Transformer (AST). Explore class
    probabilities, the spectrogram, and attention rollout
    for the most informative window.
  </p>

  <div class="chips">
    <span class="chip">AST · AudioSet pretrained</span>
    <span class="chip">ICBHI 2017</span>
    <span class="chip">4 classes</span>
    <span class="chip">8 s windows · 4 s stride</span>
  </div>
</div>
"""


NOTICE_HTML = """
<div id="research-note">
  <b>Research use only.</b>
  This prototype is not a medical device and must not be
  used for diagnosis or treatment. Predictions are
  experimental and are not a substitute for assessment
  by a qualified healthcare professional.
</div>
"""


INPUT_INFO_HTML = """
<div class="helper-grid">
  <div>
    <b>Format</b>
    WAV / librosa-readable
  </div>

  <div>
    <b>Sample rate</b>
    Resampled to 16 kHz
  </div>

  <div>
    <b>Window</b>
    8 seconds
  </div>

  <div>
    <b>Stride</b>
    4 seconds
  </div>
</div>

<div class="helper-text">
  Longer recordings are analyzed in overlapping windows
  (first 120 s). Processing time depends on recording
  length and hardware.
</div>
"""


# ============================================================
# 12. GRADIO INTERFACE
# ============================================================

blocks_args = {
    "title": "RobustLungAI | Respiratory Sound Analysis"
}

# Gradio 5 accepts theme and CSS in Blocks.
# Gradio 6 expects them in launch().
if not IS_GRADIO_6:
    blocks_args["theme"] = APP_THEME
    blocks_args["css"] = APP_CSS


with gr.Blocks(**blocks_args) as demo:

    gr.HTML(HEADER_HTML)
    gr.HTML(NOTICE_HTML)

    with gr.Row(equal_height=False):

        # ----------------------------------------------------
        # Audio input
        # ----------------------------------------------------

        with gr.Column(
            scale=4,
            min_width=300,
            elem_classes=["card"],
        ):
            gr.HTML(
                '<div class="step-title">'
                '01 · Audio input</div>'
            )

            audio_in = gr.Audio(
                type="filepath",
                sources=["upload"],
                label="Lung-sound recording",
            )

            gr.HTML(INPUT_INFO_HTML)

            with gr.Row():
                run_btn = gr.Button(
                    "Analyze recording →",
                    variant="primary",
                    elem_id="analyze-btn",
                    scale=3,
                )

                clear_btn = gr.Button(
                    "Clear",
                    elem_id="clear-btn",
                    scale=1,
                )

        # ----------------------------------------------------
        # Classification results
        # ----------------------------------------------------

        with gr.Column(
            scale=6,
            min_width=320,
            elem_classes=["card"],
        ):
            gr.HTML(
                '<div class="step-title">'
                '02 · Classification results</div>'
            )

            result_out = gr.HTML(
                EMPTY_HTML
            )

            label_out = gr.Label(
                num_top_classes=4,
                label="Class probabilities (mean over windows)",
            )

    # --------------------------------------------------------
    # Interpretability visualizations
    # --------------------------------------------------------

    with gr.Column(
        elem_classes=["card"]
    ):
        gr.HTML(
            '<div class="step-title">'
            '03 · Model interpretability</div>'
        )

        with gr.Tabs():

            with gr.Tab("Spectrogram & attention"):
                gr.Markdown(
                    "Left: time-frequency features of the "
                    "selected window. Right: attention rollout "
                    "propagated through transformer layers. "
                    "This is not a causal explanation or a "
                    "clinically validated localization map."
                )

                spec_out = gr.Image(
                    show_label=False,
                    container=False,
                    type="numpy",
                    interactive=False,
                )

            with gr.Tab("Window timeline"):
                gr.Markdown(
                    "Class probabilities for each analysis "
                    "window. The shaded band marks the window "
                    "used for the interpretability view."
                )

                timeline_out = gr.Image(
                    show_label=False,
                    container=False,
                    type="numpy",
                    interactive=False,
                )

    # --------------------------------------------------------
    # About section
    # --------------------------------------------------------

    with gr.Accordion(
        "About the model and methodology",
        open=False,
    ):
        gr.Markdown(
            """
            ### Model

            Audio Spectrogram Transformer fine-tuned for four
            respiratory sound classes using ICBHI 2017.

            ### Classes

            - **Normal** — no adventitious sounds.
            - **Crackle** — discontinuous, popping sounds.
            - **Wheeze** — continuous, high-pitched sounds.
            - **Both** — crackles and wheezes together.

            ### Recording-level inference

            Audio is band-pass filtered (50 Hz–8 kHz),
            resampled to 16 kHz, and divided into overlapping
            8-second windows with a 4-second stride.

            Displayed probabilities are the mean of
            window-level probabilities. The interpretability
            view uses the window with the highest probability
            for the predicted class.

            The model was evaluated on annotated breath cycles.
            Whole-recording aggregation is a demonstration
            convenience, not a validated diagnostic use case.

            ### Privacy

            Avoid uploading identifiable or sensitive recordings.
            Shared demos may process temporary audio files.
            """
        )

    # --------------------------------------------------------
    # Button events
    # --------------------------------------------------------

    run_btn.click(
        fn=run,
        inputs=[audio_in],
        outputs=[
            label_out,
            result_out,
            spec_out,
            timeline_out,
        ],
        show_progress="full",
    )

    clear_btn.click(
        fn=clear_all,
        inputs=None,
        outputs=[
            audio_in,
            label_out,
            result_out,
            spec_out,
            timeline_out,
        ],
    )


# ============================================================
# 13. MODEL WARM-UP
# ============================================================

print("\n" + "=" * 65)
print("Initializing RobustLungAI...")
print("=" * 65)

try:
    get_model()

except Exception:
    traceback.print_exc()

    print(
        "Model preload failed. The app will retry "
        "when the first recording is submitted."
    )


# ============================================================
# 14. LAUNCH GRADIO
# ============================================================

# Colab usually needs a share URL to open the interface.
# Set GRADIO_SHARE=0 to disable the temporary public URL.
SHARE_DEMO = (
    os.environ.get(
        "GRADIO_SHARE",
        "1",
    ).strip().lower()
    in {"1", "true", "yes"}
)

launch_args = {
    "share": SHARE_DEMO,
    "debug": False,
    "show_error": True,
}

if IS_GRADIO_6:
    launch_args["theme"] = APP_THEME
    launch_args["css"] = APP_CSS

print("\nLaunching RobustLungAI Gradio app...")

demo.queue().launch(
    **launch_args
)